In [ ]:
# データ数が少なかったので，充電した車両全体の総旅行時間を可視化

no_charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] == 0]
charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] > 0]

time = pd.to_datetime(charging_trip['StartTime'], unit='s')
charging_trip['trip_time'] = (charging_trip['EndTime'] - charging_trip['StartTime'])/ 60  # 分単位に変換
charging_trip['time_slot'] = pd.cut(time.dt.hour, bins=np.arange(0, 25, 1), right=False)
charging_trip_goal_summary = charging_trip.groupby(['time_slot', 'goalID'])['trip_time'].mean().reset_index()

time_no_charging = pd.to_datetime(no_charging_trip['StartTime'], unit='s')
no_charging_trip['trip_time'] = (no_charging_trip['EndTime'] - no_charging_trip['StartTime']) / 60  # 分単位に変換
no_charging_trip['time_slot'] = pd.cut(time_no_charging.dt.hour, bins=np.arange(0, 25, 1), right=False)
no_charging_trip_goal_summary = no_charging_trip.groupby(['time_slot', 'goalID'])['trip_time'].mean().reset_index()

# 時間帯別の旅行時間をリスト形式で準備(箱ひげ図，個々の車両旅行時間を確認したい用)
# time_slot_groups = charging_trip.groupby('time_slot')['trip_time'].apply(list)
# time_slot_groups_no_charging = no_charging_trip.groupby('time_slot')['trip_time'].apply(list)

# 時間帯別の充電時間を折れ線グラフで充電，非充電車両の旅行時間を比較
plt.figure(figsize=(12, 6))
plt.bar(charging_trip_goal_summary['time_slot'].astype(str), charging_trip_goal_summary['trip_time'], color='skyblue', label='充電車両の平均旅行時間 (分)')
plt.bar(no_charging_trip_goal_summary['time_slot'].astype(str), no_charging_trip_goal_summary['trip_time'], alpha=0.7, color='orange', label='非充電車両の平均旅行時間 (分)')
plt.title('時間帯別の平均旅行時間の比較')
plt.xlabel('時間帯')
plt.ylabel('平均旅行時間 (分)')
plt.xticks(rotation=45)
plt.grid()
plt.legend()


In [ ]:
# 充電による追加時間だけでは，充電を諦めた車両の影響の考慮ができていない。
# 充電失敗した車両の数をカウントし，その分ペナルティを加える。
# 充電失敗した車両の数をカウント
NEEDED_CHARGING_KWH = 36  # 必要な充電量(kWh).20->80%まで充電する場合の値
data_files, data_path, num_files = setting()
data_file = data_files[55]  # 最初のファイルを使用
with open(os.path.join(data_path, data_file), 'rb') as f:
    emates_result = pickle.load(f)
vehicle_trip = emates_result['vehicle_trip']
charging_loss = emates_result['charging_loss']
num_loss = len(charging_loss)
min_kw = min(emates_result['cs_config']['cap_kw'])
loss_penalty_time = NEEDED_CHARGING_KWH * num_loss / min_kw * 60 * 2 # 分単位に変換&自分の充電時間＋充電待ち時間を考慮して二倍する。
# 確実に諦めをペナルティとして学習させるために10%のマージンを取る。
loss_penalty_time *= 1.1
print(f"充電失敗による追加時間: {loss_penalty_time} 分")

In [ ]:
# 目的関数の指標の正規化。ここでは，パイロットランニングの結果を用いて、充電による追加時間を正規化する。
# 最初の50トライアル分を使用
data_files, data_path, num_files = setting()
data_files = data_files[:50]  # 最初の50ファイルを使用
results_df = aggregate_results(data_files, data_path, num_files)


### 単目的でコスト換算
- 通常時：= 初期コスト + sum(i=1~5, (運用コスト_通常時_i* + ユーザーコスト_通常時_i*365) / (1 + r)^i)

- 故障時：= 初期コスト + sum(i=1~5, (運用コスト_故障時_i* + ユーザーコスト_最悪故障時_i*365) / (1 + r)^i)

ユーザーコスト＝CSまでの移動による追加時間＋1.5 * 待ち時間

#### 故障時

In [1]:
import sys
import os
from datetime import datetime
import json
import pickle
import numpy as np
import pandas as pd
import optuna
from tqdm import tqdm
import matplotlib.pyplot as plt
import japanize_matplotlib
import shutil
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# プロジェクトのルートディレクトリをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from src.simulation.run_emates import only_run_emates
from src.simulation.data_load import save_data_to_pickle
from src.util.path_manager import get_paths
from src.util.optimization import *

import concurrent.futures
import threading
import time
from src.simulation.create_emates_env import prepare_parallel_environment


def create_failure_scenario_parallel(cs_config: dict, trial, max_workers=None):
    """故障シナリオを並列で処理する統合関数"""
    
    # 1. 設置されているCSのインデックスを取得
    installed_cs_indices = [i for i, ports in enumerate(cs_config['ports']) if ports > 0]
    
    if len(installed_cs_indices) == 0:
        return float('inf'), float('inf')
    
    print(f"\n=== Trial {trial.number}: 故障シナリオ並列処理開始 ===")
    print(f"設置CS数: {len(installed_cs_indices)}, 並列シナリオ数: {len(installed_cs_indices)}")
    
    # 2. 並列数を決定（故障シナリオ数と同じ）
    parallel_count = len(installed_cs_indices)
    if max_workers:
        parallel_count = min(parallel_count, max_workers)
    
    # 3. 並列環境を構築（各ワーカーフォルダを作成）
    print("並列環境構築中...")
    prepare_parallel_environment(cs_config, parallel_count)
    print(f"✓ {parallel_count}個のワーカー環境を構築完了")
    
    # 4. 各故障シナリオを並列で実行
    start_time = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=parallel_count) as executor:
        # 各故障シナリオのタスクを投入
        future_to_scenario = {}
        
        for worker_id, failure_cs_idx in enumerate(installed_cs_indices, 1):
            future = executor.submit(
                process_single_failure_scenario_with_worker,
                cs_config, failure_cs_idx, trial.number, SAVE_DIR, worker_id
            )
            future_to_scenario[future] = failure_cs_idx
            # print(f"故障シナリオCS{failure_cs_idx} → Worker{worker_id}に投入")
        
        # 結果を順次収集
        results = []
        completed = 0
        
        for future in concurrent.futures.as_completed(future_to_scenario):
            failure_cs_idx = future_to_scenario[future]
            
            try:
                result = future.result(timeout=3600)  # 1時間タイムアウト
                results.append(result)
                completed += 1
                
                print(f"✓ [{completed}/{len(installed_cs_indices)}] "
                      f"故障CS{failure_cs_idx}完了: "
                      f"コスト={result['cost']:.2f}万円, "
                      f"95%待ち時間={result['wait_time_95p']:.2f}秒")
                
            except concurrent.futures.TimeoutError:
                print(f"❌ 故障CS{failure_cs_idx}がタイムアウト")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': 'timeout'
                })
            except Exception as e:
                print(f"❌ 故障CS{failure_cs_idx}でエラー: {e}")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': str(e)
                })
    
    end_time = time.time()
    print(f"\n並列処理完了（{end_time - start_time:.2f}秒）")
    
    # 5. 結果の集計
    valid_results = [r for r in results if r['cost'] != float('inf')]
    
    if not valid_results:
        print("❌ 有効な結果がありません")
        return float('inf')
    
    # 最悪ケースの抽出
    worst_cost = max(r['cost'] for r in valid_results)
    worst_wait_time = max(r['wait_time_95p'] for r in valid_results)
    
    print(f"📊 結果サマリー:")
    print(f"  有効シナリオ数: {len(valid_results)}/{len(results)}")
    print(f"  最悪ケースコスト: {worst_cost:.2f}万円")
    print(f"  最悪ケース95%待ち時間: {worst_wait_time:.2f}秒")
    # ワーストケース以外を削除
    for result in results:
        if result['cost'] != worst_cost:
            # 該当ワーカーの結果ファイルを削除
            if 'results_filepath' in result:
                try:
                    os.remove(result['results_filepath'])
                    print(f"削除: {result['results_filepath']}")
                except Exception as e:
                    print(f"削除失敗: {result['results_filepath']} - {e}")

    return worst_cost

def process_single_failure_scenario_with_worker(cs_config, failure_cs_idx, trial_number, save_dir, worker_id) -> dict:
    """指定されたワーカーで単一の故障シナリオを処理"""
    
    try:
        
        # 1. 指定されたワーカー用に故障情報を作成
        create_failure_info_for_worker(cs_config, failure_cs_idx, worker_id, FAILURE_TIME=FAILURE_TIME)
        
        # 2. 該当ワーカーでシミュレーション実行（単一ワーカーのみ）
        only_run_emates(worker_id=worker_id)
        
        # 3. 結果の保存
        results_filename = f"trial_{trial_number}_failureCS_{failure_cs_idx}.pkl"
        results_filepath = os.path.join(save_dir, results_filename)
        save_data_to_pickle(worker_id=worker_id, filename=results_filepath)
        
        # 4. コスト計算
        eval_costs, _ = evaluation_total_costs(results_filepath)
        
        
        # 5. 95パーセンタイル待ち時間計算
        wait_time_95p = calculate_95percentile_wait_time(results_filepath)
        
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': eval_costs,
            'wait_time_95p': wait_time_95p,
            'results_filepath': results_filepath
        }
        
    except Exception as e:
        print(f"❌ Worker{worker_id}で故障CS{failure_cs_idx}の処理中にエラー: {e}")
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': float('inf'),
            'wait_time_95p': float('inf'),
            'error': str(e)
        }

# 並列処理版の目的関数
def cs_placement_objective_failure_parallel(trial) -> tuple:
    """並列処理版：故障時のみを考慮した多目的最適化の目的関数"""
    
    # 1. OptunaによるCS配置の提案
    cs_config = set_cs_placement(trial)
    
    # 2. CS配置の妥当性チェック
    if not check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf')
    
    # 3. 並列故障シナリオ実行
    worst_failure_cost = create_failure_scenario_parallel(
        cs_config, trial, max_workers=8  # 最大4並列に制限
    )
    
    return worst_failure_cost

def manage_pkl_files_after_optimization(study, save_dir) -> None:
    """最適化完了後のpklファイル管理（ベストトライアルのみ保持）"""
    print("\n=== 最適化完了後のファイル管理（ベストのみ保持）===")
    
    save_path = Path(save_dir)
    pkl_files = [f for f in os.listdir(save_path) if f.endswith('.pkl')]
    
    if not pkl_files:
        print("pklファイルが見つかりません。")
        return
    
    # ベストtrialを特定
    best_trial_number = study.best_trial.number
    print(f"ベストtrial: {best_trial_number}")
    
    # 保持するファイルを選定（ベストのみ）
    trials_to_keep = set()
    
    # ベストtrialのファイルパターンを検索
    best_files = [f for f in pkl_files if f.startswith(f"trial_{best_trial_number}_") or f == f"trial_{best_trial_number}.pkl"]
    
    for best_file in best_files:
        trials_to_keep.add(best_file)
        print(f"保持ファイル: {best_file}")
    
    if len(trials_to_keep) == 0:
        print(f"⚠️  ベストtrial {best_trial_number} に対応するファイルが見つかりません。")
        print("利用可能なファイル:")
        for f in pkl_files[:10]:  # 最初の10個を表示
            print(f"  {f}")
        return
    
    # 削除対象を特定
    files_to_delete = set(pkl_files) - trials_to_keep
    
    # 統計情報を表示
    print(f"\n総ファイル数: {len(pkl_files)}")
    print(f"保持ファイル数: {len(trials_to_keep)}")
    print(f"削除対象ファイル数: {len(files_to_delete)}")
    
    if len(files_to_delete) == 0:
        print("削除対象のファイルはありません。")
        return
    
    # ファイルサイズの分析
    keep_size = 0
    delete_size = 0
    
    for file_name in trials_to_keep:
        file_path = save_path / file_name
        if file_path.exists():
            keep_size += file_path.stat().st_size
    
    for file_name in files_to_delete:
        file_path = save_path / file_name
        if file_path.exists():
            delete_size += file_path.stat().st_size
    
    print(f"保持データサイズ: {keep_size / (1024**2):.1f} MB")
    print(f"削除データサイズ: {delete_size / (1024**2):.1f} MB")
    print(f"削除による節約: {delete_size / (keep_size + delete_size) * 100:.1f}%")
    
    # 自動的にバックアップディレクトリに移動
    backup_dir = save_path / "backup_deleted_trials"
    backup_dir.mkdir(exist_ok=True)
    
    moved_count = 0
    for file_name in files_to_delete:
        file_path = save_path / file_name
        backup_path = backup_dir / file_name
        
        if file_path.exists():
            try:
                shutil.move(str(file_path), str(backup_path))
                moved_count += 1
            except Exception as e:
                print(f"ファイル移動失敗 {file_name}: {e}")
    
    print(f"\n✅ {moved_count}個のファイルをバックアップディレクトリに移動しました。")
    print(f"バックアップ先: {backup_dir}")
    print(f"💾 ベストトライアル {best_trial_number} のファイルのみ保持されました。")
    
    # 残りファイルサイズを確認
    remaining_size = sum(f.stat().st_size for f in save_path.glob("*.pkl")) / (1024**2)
    print(f"残りのpklファイルサイズ: {remaining_size:.1f} MB")

# 並列処理版の最適化実行
def run_optuna_failure_optimization_parallel(n_trials=150, timeout = 60*60*24) -> None:
    """並列処理版故障時最適化の実行"""
    db_path = os.path.join(SAVE_DIR, 'optuna_failure_study_parallel.db')
    db_url = f"sqlite:///{db_path}"
    study_name = "cs_failure_optimization_parallel"
    
    try:
        study = optuna.load_study(study_name=study_name, storage=db_url)
        print(f"既存の並列処理Studyを読み込みました（トライアル数: {len(study.trials)}）")
    except KeyError:
        study = optuna.create_study(
            directions=['minimize'],
            study_name=study_name, 
            storage=db_url
        )
        print("新しい並列処理Studyを作成しました")
    
    print(f"\n🚀 並列故障シナリオ最適化を開始します")
    print(f"トライアル数: {n_trials}")
    print(f"各トライアルで故障シナリオを並列実行します")
    
    # 最適化実行
    # ✅ --- tqdmによるプログレスバーのセットアップ ---
    with tqdm(total=n_trials, desc="Optimization Progress") as pbar:
        # 各トライアル完了時にプログレスバーを更新するコールバック関数
        def progress_bar_callback(study, trial):
            pbar.update(1)
        study.optimize(cs_placement_objective_failure_parallel, n_trials=n_trials,
                       callbacks=[progress_bar_callback], timeout=timeout)
    # ✅ --- プログレスバーのセットアップここまで ---
    manage_pkl_files_after_optimization(study, SAVE_DIR)


In [2]:
current_time = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = current_time
SAVE_DIR = f'{SAVE_DIR}_1DAY'
# SAVE_DIR = '20250820_1819_7PM'
os.makedirs(SAVE_DIR, exist_ok=True)
FAILURE_FLAG = True
FAILURE_TIME = list(range(0, 25))
run_optuna_failure_optimization_parallel(n_trials=500, timeout=60*60*28)  # 1日以内に完了するように設定

[I 2025-08-22 19:16:18,752] A new study created in RDB with name: cs_failure_optimization_parallel


新しい並列処理Studyを作成しました

🚀 並列故障シナリオ最適化を開始します
トライアル数: 500
各トライアルで故障シナリオを並列実行します


Optimization Progress:   0%|          | 0/500 [00:00<?, ?it/s]


=== Trial 0: 故障シナリオ並列処理開始 ===
設置CS数: 4, 並列シナリオ数: 4
並列環境構築中...
✓ 4個のワーカー環境を構築完了
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
✓ [1/4] 故障CS5完了: コスト=-5941.43万円, 95%待ち時間=1231.59秒
✓ [2/4] 故障CS2完了: コスト=-4613.26万円, 95%待ち時間=1510.18秒
✓ [3/4] 故障CS7完了: コスト=-1267.11万円, 95%待ち時間=1954.60秒


[I 2025-08-22 19:22:05,535] Trial 0 finished with value: 4355.015447888178 and parameters: {}. Best is trial 0 with value: 4355.015447888178.
Optimization Progress:   0%|          | 1/500 [05:46<48:04:04, 346.78s/it]

...

✓ [5/5] 故障CS0完了: コスト=-889.90万円, 95%待ち時間=1855.19秒

並列処理完了（379.38秒）
📊 結果サマリー:
  有効シナリオ数: 5/5
  最悪ケースコスト: -889.90万円
  最悪ケース95%待ち時間: 1855.19秒
削除: 20250822_1916_1DAY\trial_258_failureCS_6.pkl
削除: 20250822_1916_1DAY\trial_258_failureCS_5.pkl
削除: 20250822_1916_1DAY\trial_258_failureCS_2.pkl
削除: 20250822_1916_1DAY\trial_258_failureCS_7.pkl

=== Trial 259: 故障シナリオ並列処理開始 ===
設置CS数: 5, 並列シナリオ数: 5
並列環境構築中...
✓ 5個のワーカー環境を構築完了
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39

[I 2025-08-23 23:11:27,926] Trial 259 finished with value: -3241.416027176907 and parameters: {'ports_0': 3, 'capacity_0': 100, 'ports_1': 0, 'ports_2': 1, 'capacity_2': 50, 'ports_3': 0, 'ports_4': 0, 'ports_5': 1, 'capacity_5': 100, 'ports_6': 3, 'capacity_6': 100, 'ports_7': 1, 'capacity_7': 100}. Best is trial 61 with value: -3307.4034791659215.
Optimization Progress:  52%|█████▏    | 260/500 [27:55:09<25:11:06, 377.78s/it]

✓ [5/5] 故障CS0完了: コスト=-3241.42万円, 95%待ち時間=1278.76秒

並列処理完了（379.27秒）
📊 結果サマリー:
  有効シナリオ数: 5/5
  最悪ケースコスト: -3241.42万円
  最悪ケース95%待ち時間: 1514.79秒
削除: 20250822_1916_1DAY\trial_259_failureCS_7.pkl
削除: 20250822_1916_1DAY\trial_259_failureCS_6.pkl
削除: 20250822_1916_1DAY\trial_259_failureCS_5.pkl
削除: 20250822_1916_1DAY\trial_259_failureCS_2.pkl

=== Trial 260: 故障シナリオ並列処理開始 ===
設置CS数: 6, 並列シナリオ数: 6
並列環境構築中...
✓ 6個のワーカー環境を構築完了
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 39600, 43200, 46800, 50400, 54000, 57600, 61200, 64800, 68400, 72000, 75600, 79200, 82800, 86400]
[0, 3600, 7200, 10800, 14400, 18000, 21600, 25200, 28800, 32400, 36000, 

[I 2025-08-23 23:18:11,442] Trial 260 finished with value: 2418.779538201972 and parameters: {'ports_0': 3, 'capacity_0': 100, 'ports_1': 0, 'ports_2': 1, 'capacity_2': 50, 'ports_3': 0, 'ports_4': 2, 'capacity_4': 100, 'ports_5': 1, 'capacity_5': 100, 'ports_6': 3, 'capacity_6': 100, 'ports_7': 1, 'capacity_7': 100}. Best is trial 61 with value: -3307.4034791659215.
Optimization Progress:  52%|█████▏    | 261/500 [28:01:52<25:40:06, 386.64s/it]

✓ [6/6] 故障CS5完了: コスト=2418.78万円, 95%待ち時間=1404.18秒

並列処理完了（403.34秒）
📊 結果サマリー:
  有効シナリオ数: 6/6
  最悪ケースコスト: 2418.78万円
  最悪ケース95%待ち時間: 1609.09秒
削除: 20250822_1916_1DAY\trial_260_failureCS_7.pkl
削除: 20250822_1916_1DAY\trial_260_failureCS_4.pkl
削除: 20250822_1916_1DAY\trial_260_failureCS_2.pkl
削除: 20250822_1916_1DAY\trial_260_failureCS_0.pkl
削除: 20250822_1916_1DAY\trial_260_failureCS_6.pkl

=== 最適化完了後のファイル管理（ベストのみ保持）===
ベストtrial: 61
保持ファイル: trial_61_failureCS_0.pkl

総ファイル数: 261
保持ファイル数: 1
削除対象ファイル数: 260
保持データサイズ: 5.2 MB
削除データサイズ: 1361.1 MB
削除による節約: 99.6%

✅ 260個のファイルをバックアップディレクトリに移動しました。
バックアップ先: 20250822_1916_1DAY\backup_deleted_trials
💾 ベストトライアル 61 のファイルのみ保持されました。
残りのpklファイルサイズ: 5.2 MB


In [ ]:
import optuna
import os
from pathlib import Path
import shutil
import pickle

SAVE_DIR = '../20250815_0955'
study = optuna.load_study(study_name="cs_optimization", storage=f"sqlite:///{SAVE_DIR}/optuna_study.db")

def manage_pkl_files_after_optimization(study, save_dir):
    """最適化完了後のpklファイル管理（ベストトライアルのみ保持）"""
    print("\n=== 最適化完了後のファイル管理（ベストのみ保持）===")
    
    save_path = Path(save_dir)
    pkl_files = [f for f in os.listdir(save_path) if f.endswith('.pkl')]
    
    if not pkl_files:
        print("pklファイルが見つかりません。")
        return
    
    # ベストtrialを特定
    best_trial_number = study.best_trial.number
    print(f"ベストtrial: {best_trial_number}")
    
    # 保持するファイルを選定（ベストのみ）
    trials_to_keep = set()
    
    # ベストtrialのファイルパターンを検索
    best_files = [f for f in pkl_files if f.startswith(f"trial_{best_trial_number}_") or f == f"trial_{best_trial_number}.pkl"]
    
    for best_file in best_files:
        trials_to_keep.add(best_file)
        print(f"保持ファイル: {best_file}")
    
    if len(trials_to_keep) == 0:
        print(f"⚠️  ベストtrial {best_trial_number} に対応するファイルが見つかりません。")
        print("利用可能なファイル:")
        for f in pkl_files[:10]:  # 最初の10個を表示
            print(f"  {f}")
        return
    
    # 削除対象を特定
    files_to_delete = set(pkl_files) - trials_to_keep
    
    # 統計情報を表示
    print(f"\n総ファイル数: {len(pkl_files)}")
    print(f"保持ファイル数: {len(trials_to_keep)}")
    print(f"削除対象ファイル数: {len(files_to_delete)}")
    
    if len(files_to_delete) == 0:
        print("削除対象のファイルはありません。")
        return
    
    # ファイルサイズの分析
    keep_size = 0
    delete_size = 0
    
    for file_name in trials_to_keep:
        file_path = save_path / file_name
        if file_path.exists():
            keep_size += file_path.stat().st_size
    
    for file_name in files_to_delete:
        file_path = save_path / file_name
        if file_path.exists():
            delete_size += file_path.stat().st_size
    
    print(f"保持データサイズ: {keep_size / (1024**2):.1f} MB")
    print(f"削除データサイズ: {delete_size / (1024**2):.1f} MB")
    print(f"削除による節約: {delete_size / (keep_size + delete_size) * 100:.1f}%")
    
    # 自動的にバックアップディレクトリに移動
    backup_dir = save_path / "backup_deleted_trials"
    backup_dir.mkdir(exist_ok=True)
    
    moved_count = 0
    for file_name in files_to_delete:
        file_path = save_path / file_name
        backup_path = backup_dir / file_name
        
        if file_path.exists():
            try:
                shutil.move(str(file_path), str(backup_path))
                moved_count += 1
            except Exception as e:
                print(f"ファイル移動失敗 {file_name}: {e}")
    
    print(f"\n✅ {moved_count}個のファイルをバックアップディレクトリに移動しました。")
    print(f"バックアップ先: {backup_dir}")
    print(f"💾 ベストトライアル {best_trial_number} のファイルのみ保持されました。")
    
    # 残りファイルサイズを確認
    remaining_size = sum(f.stat().st_size for f in save_path.glob("*.pkl")) / (1024**2)
    print(f"残りのpklファイルサイズ: {remaining_size:.1f} MB")
manage_pkl_files_after_optimization(study, SAVE_DIR)